# 01 · Data (Yahoo Finance and Kenneth French)

* **Calibration panel** (H1, H2, H4): equity indices, US sector ETFs, country ETFs, bond ETFs, FX and futures from
  Yahoo Finance (adjusted daily closes), plus ^VIX for the volatility-regime split. Crypto is downloaded as an
  exploratory extension only.
* **Factor panels** (H3): Kenneth French daily factors: US (Mkt-RF, SMB, HML from 1926; RMW, CMA from 1963; Mom)
  and developed-market regions (Developed, North America, Europe, Japan, Asia-Pacific ex Japan; 5 factors + WML).

In development mode the Yahoo request stops at `dev_end` and the French files are cut at each panel's `dev_end`
before being written. Downloads are cached in chunks, so re-running after a disconnect resumes.

**Two runs, same notebooks.** First run everything with `project.mode: development` (data end at each panel's
`dev_end`; holdout dates never enter any file). After both authors sign off the design and the code is frozen, set
`mode: holdout`, `signoff: true` and `frozen_commit: <hash>` in `config/config.yaml` and run 01→06 once more. The
holdout run reuses γ and the CVM level chosen in development and stops if the configuration has changed.

In [ ]:
# ---------------------------------------------------------------- setup (run this cell first)
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/CalVol_project"   # <- change if you placed the folder elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)
else:
    PROJECT_DIR = os.path.abspath("..")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import json
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from calvol.utils import load_config, get_paths, check_gate
from calvol import pipeline as pp
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 220)

cfg = load_config(PROJECT_DIR)          # mode / sign-off / smoke test are set in config/config.yaml (project: ...)
MODE = check_gate(cfg)                  # refuses a holdout run without sign-off and a frozen commit
P = get_paths(cfg)
PER = pp.eval_split(cfg)
print(f"MODE = {MODE.upper()}  |  evaluation period = {PER}  |  smoke test = {cfg['project']['smoke_test']}")
print("Results folder:", P.results)

def show(name):
    f = P.figures / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=760))
    else:
        print("(figure not produced:", name, ")")

def table(name, round_=4):
    f = P.tables / f"{name}.csv"
    if f.exists():
        display(pd.read_csv(f).round(round_))
    else:
        print("(table not produced:", name, ")")

In [ ]:
info = pp.stage_data(cfg)
print(json.dumps(info, indent=2, default=str))

## Download reports
Check that every instrument has the expected history; `yahoo_missing` above lists symbols Yahoo returned nothing for.

In [ ]:
rep = pd.read_csv(P.tables / "download_report.csv")
display(rep.groupby("class").agg(instruments=("ticker", "size"), first=("min", "min"), last=("max", "max")))
display(rep.sort_values("min").tail(15))
display(pd.read_csv(P.tables / "french_factors_report.csv"))